# ANLP Assignment 1 — Train the C1–C5 Ablation Locally (RTX 3050, 4 GB)

This notebook runs the **entire assignment end-to-end on your own laptop** — no Kaggle,
no Colab. It trains all five configurations, evaluates them with greedy decoding, logs
everything to Weights & Biases, and writes the plots/tables the report needs.

**Companion notebook:** `02_keys_wandb_hf_kaggle.ipynb` handles API keys and pushes the
finished checkpoints to Hugging Face and Kaggle. Run this one first.

---

### What gets built

| Config | Positional | Attention | Norm | Tokenization |
|---|---|---|---|---|
| **C1** (base) | Sinusoidal absolute | Multi-Head | LayerNorm | BPE subword |
| **C2** | **RoPE** | Multi-Head | LayerNorm | BPE subword |
| **C3** | Sinusoidal absolute | **Grouped-Query** | LayerNorm | BPE subword |
| **C4** | Sinusoidal absolute | Multi-Head | **RMSNorm** | BPE subword |
| **C5** | Sinusoidal absolute | Multi-Head | LayerNorm | **BLT (token-free, entropy-patched bytes)** |

Each of C2–C5 changes **exactly one** component from C1, which is what makes the
ablation controlled.

### Measured runtime on this machine (RTX 3050 Laptop, 4 GB, bf16)

| Config | s / epoch | 25 epochs | Peak GPU |
|---|---|---|---|
| C1–C4 (tokenized) | ~138 s | ~58 min each | ~0.4 GB |
| C5 (BLT, byte-level) | ~278 s | ~2 h | ~0.7 GB |
| **Total** | | **≈ 5.5–6 h** | fits 4 GB with room to spare |

Early stopping (patience 5) often finishes sooner. Leave it running overnight.

---
## Step 0 — Environment

> ### ⚠️ The single most important step
> A default `pip install torch` on Windows installs the **CPU-only** build, which
> silently ignores your RTX 3050 and makes this notebook ~30× slower.
> You need the **CUDA** build. Check below; if it says `cuda available: False`,
> run the install cell that follows.

In [ ]:
import subprocess, sys, os, platform

print("Python  :", sys.version.split()[0])
print("Platform:", platform.platform())
print()

# --- NVIDIA driver ---------------------------------------------------------
try:
    out = subprocess.run(["nvidia-smi",
                          "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=30)
    print("nvidia-smi:", out.stdout.strip() or out.stderr.strip())
except FileNotFoundError:
    print("nvidia-smi NOT FOUND -> install the NVIDIA driver from nvidia.com/drivers")

# --- PyTorch ---------------------------------------------------------------
try:
    import torch
    print("torch          :", torch.__version__)
    print("cuda available :", torch.cuda.is_available())
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print("device         :", p.name)
        print("VRAM           : %.2f GB" % (p.total_memory / 1024**3))
        print("bf16 supported :", torch.cuda.is_bf16_supported())
    else:
        print("\n>>> CPU-ONLY BUILD DETECTED - run the next cell. <<<")
except ImportError:
    print("torch not installed - run the next cell.")

### 0a. Install the CUDA build of PyTorch (only if the check above said `False`)

Your driver (571.96) supports CUDA 12.8, so the `cu126` wheels are a safe match.
This downloads ~2.5 GB and takes 5–15 minutes.

**Uncomment and run**, then **restart the kernel** afterwards.

In [ ]:
# !{sys.executable} -m pip install --upgrade torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu126

# After it finishes:  Kernel -> Restart Kernel, then re-run the check cell above.

### 0b. The remaining dependencies

In [ ]:
# !{sys.executable} -m pip install -q wandb huggingface_hub kagglehub kaggle pandas numpy matplotlib tqdm

import importlib
for m in ["numpy", "matplotlib", "tqdm", "wandb", "huggingface_hub"]:
    try:
        importlib.import_module(m); print(f"  ok   {m}")
    except ImportError:
        print(f"  MISSING {m}  <- uncomment the pip line above")

### 0c. Point the notebook at the project folder

Everything below is relative to `PROJECT`. Change it if you moved the folder.

In [ ]:
import os, sys, json, subprocess, textwrap
from pathlib import Path

PROJECT = Path(r"c:\Users\vidva\OneDrive - International Institute of Information Technology\Desktop\Course_Work_4-1\ANLP\Assignments\Assignment1")

assert PROJECT.exists(), f"PROJECT not found: {PROJECT}"
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

print("working dir:", os.getcwd())
for p in ["src", "src/models", "data", "outputs"]:
    print(f"  {'ok  ' if (PROJECT/p).exists() else 'MISS'} {p}")


def run(cmd, title=""):
    # Run a command and stream its output straight into the notebook.
    if title:
        print("=" * 78); print(title); print("=" * 78, flush=True)
    proc = subprocess.Popen(cmd, cwd=str(PROJECT), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True,
                            encoding="utf-8", errors="replace", bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    print(f"\n[exit code {proc.returncode}]")
    return proc.returncode

---
## Step 1 — Look at the data before modelling it

`data/brown_cipher.txt` and `data/brown_plain.txt` are 5,000 strictly line-aligned
pairs. Every plaintext character maps to **exactly 8 cipher bits**, so the two files
line up at a fixed 8:1 ratio.

In [ ]:
plain  = (PROJECT / "data" / "brown_plain.txt").read_text(encoding="utf-8").splitlines()
cipher = (PROJECT / "data" / "brown_cipher.txt").read_text(encoding="utf-8").splitlines()

print(f"lines            : {len(plain)} plain / {len(cipher)} cipher")
print(f"plain  chars     : min {min(map(len,plain))}, max {max(map(len,plain))}, "
      f"mean {sum(map(len,plain))/len(plain):.0f}")
print(f"cipher chars     : min {min(map(len,cipher))}, max {max(map(len,cipher))}, "
      f"mean {sum(map(len,cipher))/len(cipher):.0f}")
print(f"cipher alphabet  : {sorted(set(''.join(cipher[:200])))}")
print(f"len(cipher)/len(plain) constant 8? "
      f"{all(len(c) == 8*len(p) for p, c in zip(plain, cipher))}")
print()
print("plain [0][:70] :", repr(plain[0][:70]))
print("cipher[0][:70] :", repr(cipher[0][:70]))

### 1a. What is the cipher, actually?

Worth knowing so you can defend the results in the viva. Read the cipher 8 bits at a
time to get a byte, XOR it against the corresponding plaintext character, and look at
what falls out.

**The model is never told this** — it has to learn the mapping from data alone. But it
tells us the task *is* perfectly learnable, and that it is **position-dependent with
period 8**, which is exactly why the positional-encoding ablation (C1 vs C2) is
meaningful here.

In [ ]:
def xor_stream(p_line, c_line, n=32):
    out = []
    for i, ch in enumerate(p_line[:n]):
        byte = int(c_line[i*8:(i+1)*8], 2)
        out.append(byte ^ ord(ch))
    return out

for li in (0, 1, 2, 1234):
    s = xor_stream(plain[li], cipher[li], 24)
    print(f"line {li:>5}: {s[:16]}  ->  {''.join(chr(x) for x in s[:16])!r}")

print("\nThe XOR stream is the repeating 8-byte key 'ANLP2026', restarting at every line.")
print("So: cipher_byte[i] = ord(plain[i]) XOR key[i mod 8],  written MSB-first as 8 bits.")

### 1b. Why we train on 32-character segments

Full lines average 598 plaintext characters ⇒ **4,781 cipher bits**. Full attention is
O(T²), so a 4,781-token encoder will not fit in 4 GB, and the longest line (21,360 bits)
would not fit anywhere.

`--chunk-chars 32` cuts each *document* into aligned 32-character segments **after** the
train/val/test split, so no document ever straddles two splits (no leakage). 32 is a
multiple of the key period 8, so every segment starts at key phase 0 and stays solvable.

This yields **76,823 / 9,461 / 9,544** train/val/test segments.

In [ ]:
n_seg = sum(-(-len(p) // 32) for p in plain)
print(f"32-char segments over the whole corpus: {n_seg:,}")
print(f"source bits per segment                : {32*8}")
print(f"BPE compresses that to                 : ~23 tokens (see the [tok] line at train time)")

### A note on how C5 processes the cipher

Two details are easy to get wrong and both are implemented explicitly:

1. **Bytes, not the text of bytes.** The ciphertext is a *binary sequence* stored as
   ASCII `0`/`1` characters. Every **8 bits are packed into one byte value 0-255**, so a
   256-character segment is **32 bytes**, not 256. Feeding the `0`/`1` characters would
   process the text encoding of the data and inflate the sequence eightfold.

2. **Entropy-based dynamic patching, not fixed width.** Patch boundaries open where
   `H(x_t | x_{t-2}, x_{t-1}) > theta`, with the entropy from a lightweight order-2 byte
   n-gram model fit on the training split only (`src/models/entropy.py`). No separate
   neural LM is trained. `theta` is calibrated to a target mean patch length, giving
   about 10 variable-length patches per 34-byte sequence.

Run `--blt-fixed-patches` to switch back to fixed-width patches as an ablation.


---
## Step 2 — Unit-test the from-scratch modules

Before spending six hours training, check that the hand-written components are actually
correct. `tests/test_modules.py` verifies, among other things:

* scaled dot-product attention against an explicit loop,
* that `LayerNorm` / `RMSNorm` match reference formulas,
* that RoPE really makes attention logits depend only on **relative** distance,
* that GQA with `n_kv_heads == n_heads` reproduces MHA exactly,
* that the decoder is **causal** (perturbing future tokens cannot change earlier logits),
* that the BLT stack is causal under **entropy-based dynamic patching**, that a
  patch boundary never depends on the byte it precedes, and that appending bytes
  cannot renumber earlier patches (greedy decoding depends on both),
* that 8 cipher bits are packed into one byte, losslessly.

In [ ]:
rc = run([sys.executable, "-m", "pytest", "tests/test_modules.py", "-q"],
         "Unit tests for the from-scratch modules")
if rc != 0:
    print("\nIf pytest is missing:  !{sys.executable} -m pip install pytest")

---
## Step 3 — 2-minute pipeline smoke test

A tiny end-to-end run (200 samples, 1 epoch, W&B off) that touches every code path:
data loading → chunking → BPE training → model build → train step → greedy decode →
metrics → checkpoint. If this passes, the six-hour run will not die at hour five.

In [ ]:
rc = run([sys.executable, "-m", "src.train",
          "--config", "C1",
          "--src-file", "data/brown_cipher.txt",
          "--tgt-file", "data/brown_plain.txt",
          "--chunk-chars", "32",
          "--smoke", "--wandb", "disabled",
          "--out-dir", "outputs_smoke"],
         "SMOKE TEST - C1")

In [ ]:
# Same, for the BLT path (C5) - it exercises completely different modules.
rc = run([sys.executable, "-m", "src.train",
          "--config", "C5",
          "--src-file", "data/brown_cipher.txt",
          "--tgt-file", "data/brown_plain.txt",
          "--chunk-chars", "32",
          "--smoke", "--wandb", "disabled",
          "--out-dir", "outputs_smoke"],
         "SMOKE TEST - C5 (BLT)")

---
## Step 4 — W&B login

The assignment requires W&B logs. Run `02_keys_wandb_hf_kaggle.ipynb` **Step 1–3** first
if you have not put your key anywhere yet; the cell below just confirms it worked.

If you would rather train without logging, set `--wandb disabled` in Step 5.

In [ ]:
# load .env (created by setup_keys.py / notebook 02) into this process
env_path = PROJECT / ".env"
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            if v.strip():
                os.environ.setdefault(k.strip(), v.strip())

import wandb
try:
    ok = wandb.login(key=os.environ.get("WANDB_API_KEY") or None)
    print("wandb login:", "OK" if ok else "FAILED")
    print("entity     :", wandb.Api().viewer.username)
except Exception as e:
    print("wandb login failed:", e)
    print("-> open 02_keys_wandb_hf_kaggle.ipynb and follow Step 1.")

---
## Step 5 — Run the full ablation  ⏳ ~5.5–6 hours

`run_ablation.py` launches `src/run_all.py`, which trains **each configuration in its own
subprocess**. That is deliberate: `torch.cuda.max_memory_allocated()` is a per-process
high-water mark, so running them separately is the only way the C1-vs-C5 peak-memory
comparison means anything.

All five configs share identical depth, width, LR, batch size, schedule and seed —
only the one ablated component changes.

> **Tip:** if the kernel might disconnect, run this in a terminal instead:
> ```
> cd "<project folder>"
> python run_ablation.py > outputs/train_log.txt 2>&1
> ```
> then use Step 6 to read the results back.

In [ ]:
# The whole thing. Grab a coffee / go to sleep.
rc = run([sys.executable, "-u", "run_ablation.py"],
         "FULL ABLATION  C1 -> C5")

**Re-running individual configs.** If one config failed or you want to tweak it:

```python
run([sys.executable, "-u", "run_ablation.py", "--configs", "C3"])
run([sys.executable, "-u", "run_ablation.py", "--skip-existing"])   # resume
run([sys.executable, "-u", "run_ablation.py", "--aggregate-only"])  # rebuild tables/plots
```

---
## Step 6 — Results

In [ ]:
import json
res_path = PROJECT / "outputs" / "all_results.json"
assert res_path.exists(), "No results yet - finish Step 5 (or run with --aggregate-only)."
R = json.loads(res_path.read_text(encoding="utf-8"))
print("configs:", ", ".join(sorted(R)))

def table(cols, headers, fmt="{:.2f}"):
    w = max(len(h) for h in headers)
    line = f"{'Config':<8}" + "".join(f"{h:>16}" for h in headers)
    print(line); print("-" * len(line))
    for n in sorted(R):
        row = f"{n:<8}"
        for c in cols:
            v = R[n].get(c)
            row += f"{(fmt.format(v) if isinstance(v,(int,float)) else str(v)):>16}"
        print(row)

print("\n### Quality (greedy decoding, test split)\n")
table(["bit_accuracy", "sequence_accuracy", "levenshtein", "bleu", "rougeL"],
      ["Bit Acc %", "Seq Acc %", "Levenshtein", "BLEU", "ROUGE-L"])

print("\n### Teacher-forced\n")
table(["test_loss", "test_ppl", "test_token_accuracy", "best_epoch"],
      ["Test loss", "Test PPL", "Token Acc %", "Best epoch"])

print("\n### Cost\n")
table(["params_millions", "sec_per_epoch", "peak_gpu_mem_mb", "generation_seconds"],
      ["Params (M)", "s/epoch", "Peak GPU MB", "Decode (s)"])

In [ ]:
from IPython.display import Image, Markdown, display

md_path = PROJECT / "outputs" / "report_tables.md"
if md_path.exists():
    display(Markdown(md_path.read_text(encoding="utf-8")))

In [ ]:
for png in ["ablation_quality.png", "ablation_efficiency.png"]:
    p = PROJECT / "outputs" / png
    if p.exists():
        print(png); display(Image(str(p)))

for c in ["C1", "C2", "C3", "C4", "C5"]:
    p = PROJECT / "outputs" / c / f"{c}_curves.png"
    if p.exists():
        print(f"{c} training curves"); display(Image(str(p)))

In [ ]:
# A few qualitative decryptions from the best model
import json, random
best = max(R, key=lambda n: R[n].get("bit_accuracy", 0))
p = PROJECT / "outputs" / best / "predictions_test.json"
if p.exists():
    preds = json.loads(p.read_text(encoding="utf-8"))
    rows = preds if isinstance(preds, list) else preds.get("samples", [])
    print(f"--- {best} (best bit accuracy) ---\n")
    for r in rows[:8]:
        ref = r.get("target") or r.get("reference") or r.get("tgt", "")
        hyp = r.get("prediction") or r.get("pred") or r.get("hyp", "")
        print(f"  ref : {ref!r}\n  pred: {hyp!r}\n")

---
## Step 7 — Publish

Everything you need is now under `outputs/`. Move to
**`02_keys_wandb_hf_kaggle.ipynb`** to:

1. push the five checkpoints to the **Hugging Face Hub**,
2. push them to **Kaggle** (Models + Dataset),
3. collect the **W&B run URLs** for the report,
4. build the submission ZIP.